In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression as lr 
import matplotlib.pyplot as plt
import statsmodels.api as sm

## Data:

In [2]:
path = r"C:\Users\oskar\OneDrive\Desktop\4 Semester\Dataproject\Federated-dental-risk-vol2\federated-dental-risk-prediction\data\raw\synthetic_dataset_A_non-iid.csv"
df = pd.read_csv(path)

# Vi standardisere vores features.
variables = df.columns[1:27].tolist()
scaler = StandardScaler()
df[variables] = scaler.fit_transform(df[variables])

# Test Data:
path_testdata = r"C:\Users\oskar\OneDrive\Desktop\4 Semester\Dataproject\Federated-dental-risk-vol2\federated-dental-risk-prediction\data\processed\A\global_test_set_non-iid.csv"
test_data = pd.read_csv(path_testdata)
test_data[variables] = scaler.transform(test_data[variables])

## Functions:

In [6]:
# Backward:
def backward_elimination(X, y, alpha=0.05, verbose=True):
    """
    Backward elimination baseret på p-værdier.
    
    Parametre
    ---------
    X : pd.DataFrame
        Forklarende variable.
    y : pd.Series eller array-lignende
        Responsvariabel.
    alpha : float
        Signifikansniveau. Variable med p-værdi > alpha fjernes.
    verbose : bool
        Hvis True, printes hvilke variable der fjernes undervejs.
    
    Returnerer
    ----------
    selected_features : list
        Navne på de features der er tilbage.
    final_model : statsmodels-regression-resultat
        Den endelige fitted model.
    """
    
    X_selected = X.copy()
    
    while True:
        X_with_const = sm.add_constant(X_selected, has_constant="add")
        model = sm.OLS(y, X_with_const).fit()
        
        p_values = model.pvalues.drop("const", errors="ignore")
        max_p_value = p_values.max()
        
        if max_p_value > alpha:
            feature_to_remove = p_values.idxmax()
            if verbose:
                print(f"Fjerner '{feature_to_remove}' med p-værdi = {max_p_value:.4f}")
            X_selected = X_selected.drop(columns=[feature_to_remove])
        else:
            break
    
    final_X = sm.add_constant(X_selected, has_constant="add")
    final_model = sm.OLS(y, final_X).fit()
    
    return list(X_selected.columns), final_model

def forward_selection(X, y, alpha=0.05, verbose=True):
    """
    Forward selection baseret på p-værdier.

    Parametre
    ---------
    X : pd.DataFrame
        Forklarende variable.
    y : pd.Series eller array-lignende
        Responsvariabel.
    alpha : float
        Signifikansniveau. En variabel tilføjes kun hvis p-værdien < alpha.
    verbose : bool
        Hvis True, printes hvilke variable der tilføjes undervejs.

    Returnerer
    ----------
    selected_features : list
        Navne på de features der er valgt.
    final_model : statsmodels-regression-resultat
        Den endelige fitted model.
    """

    remaining_features = list(X.columns)
    selected_features = []

    while remaining_features:
        pvals = {}

        for feature in remaining_features:
            features_to_test = selected_features + [feature]
            X_model = sm.add_constant(X[features_to_test], has_constant="add")
            model = sm.OLS(y, X_model).fit()
            pvals[feature] = model.pvalues[feature]

        best_feature = min(pvals, key=pvals.get)
        best_pval = pvals[best_feature]

        if best_pval < alpha:
            selected_features.append(best_feature)
            remaining_features.remove(best_feature)
            if verbose:
                print(f"Tilføjer '{best_feature}' med p-værdi = {best_pval:.4f}")
        else:
            break

    final_X = sm.add_constant(X[selected_features], has_constant="add")
    final_model = sm.OLS(y, final_X).fit()

    return selected_features, final_model

## Results:

### Backward Centralized:

In [5]:
features, model = backward_elimination(df[variables], df["Risk_AlveolarOsteitis"])
print(model.params)

Fjerner 'Cyst' med p-værdi = 0.9883
Fjerner 'Prev_Extraction_Issue' med p-værdi = 0.9628
Fjerner 'Tooth_Mobility' med p-værdi = 0.9448
Fjerner 'Pain' med p-værdi = 0.9347
Fjerner 'Osteoporosis' med p-værdi = 0.7625
Fjerner 'Periodontal_Status' med p-værdi = 0.7333
Fjerner 'Sex' med p-værdi = 0.6148
Fjerner 'Root_Count' med p-værdi = 0.6163
Fjerner 'Root_Curvature' med p-værdi = 0.5963
Fjerner 'Diabetes' med p-værdi = 0.5716
Fjerner 'Patient' med p-værdi = 0.4963
Fjerner 'Tooth_Angulation' med p-værdi = 0.3981
Fjerner 'Root_Development' med p-værdi = 0.3157
Fjerner 'Bone_Density' med p-værdi = 0.2779
Fjerner 'Swelling' med p-værdi = 0.1557
Fjerner 'Smoking' med p-værdi = 0.1395
Fjerner 'Clotting_Disorder' med p-værdi = 0.1330
Fjerner 'Caries_Wisdom' med p-værdi = 0.0724
Fjerner 'Bisphosphonates' med p-værdi = 0.0551
const                       0.088872
Age                         0.012257
Trismus                     0.004064
Pericoronitis               0.020224
Caries_Adjacent          

### Forward Centralized:

In [8]:
features, model = forward_selection(df[variables], df["Risk_AlveolarOsteitis"])
print(model.params)

Tilføjer 'Proximity_Nerve' med p-værdi = 0.0000
Tilføjer 'Pericoronitis' med p-værdi = 0.0000
Tilføjer 'Impaction_Depth' med p-værdi = 0.0000
Tilføjer 'Age' med p-værdi = 0.0000
Tilføjer 'Surgical_Extraction_Type' med p-værdi = 0.0000
Tilføjer 'Trismus' med p-værdi = 0.0016
Tilføjer 'Caries_Adjacent' med p-værdi = 0.0252
const                       0.088872
Proximity_Nerve             0.037978
Pericoronitis               0.020224
Impaction_Depth             0.016990
Age                         0.012257
Surgical_Extraction_Type    0.007296
Trismus                     0.004064
Caries_Adjacent            -0.002893
dtype: float64
